# Burhan Datasets - Upload Verification & Test

This notebook verifies that uploaded datasets are accessible and working correctly.

## Prerequisites
- Google Colab (free T4 GPU recommended for embedding later)
- Hugging Face dataset uploaded

## Usage
1. Upload this notebook to Colab
2. Run all cells
3. Check verification results

In [ ]:
# Install required packages
!pip install datasets huggingface_hub -q

## 1. Configuration

In [ ]:
REPO_ID = "Kandil7/Burhan-Datasets"

COLLECTIONS = {
    "fiqh_passages": "Islamic jurisprudence",
    "hadith_passages": "Prophetic traditions",
    "quran_tafsir": "Quran interpretation",
    "aqeedah_passages": "Islamic creed",
    "seerah_passages": "Prophet biography",
    "islamic_history_passages": "Islamic history",
    "arabic_language_passages": "Arabic language",
    "spirituality_passages": "Spirituality",
    "general_islamic": "General Islamic knowledge",
    "usul_fiqh": "Principles of jurisprudence",
}

print(f"Testing repository: {REPO_ID}")
print(f"Collections: {len(COLLECTIONS)}")

## 2. Test Loading Collections

In [ ]:
from datasets import load_dataset
import json

results = {}

for collection, description in COLLECTIONS.items():
    try:
        print(f"\n📖 Loading {collection}...")
        
        # Try loading from Hugging Face
        ds = load_dataset(
            "json",
            data_files=f"hf://datasets/{REPO_ID}/collections/{collection}.jsonl.gz",
            split="train"
        )
        
        results[collection] = {
            "status": "✅ OK",
            "size": len(ds),
            "description": description,
        }
        
        print(f"  ✅ Loaded {len(ds):,} documents")
        
        # Show sample
        sample = ds[0]
        print(f"  📄 Sample keys: {list(sample.keys())}")
        
    except Exception as e:
        results[collection] = {
            "status": f"❌ Error: {str(e)[:50]}",
            "size": 0,
            "description": description,
        }
        print(f"  ❌ Failed: {e}")

## 3. Verification Summary

In [ ]:
import pandas as pd

# Create summary table
df = pd.DataFrame(results).T
df.index.name = "Collection"
print("\n" + "=" * 70)
print("📊 VERIFICATION SUMMARY")
print("=" * 70)
print(df.to_string())
print("=" * 70)

# Statistics
total_docs = sum(r["size"] for r in results.values())
successful = sum(1 for r in results.values() if "OK" in r["status"])
failed = len(results) - successful

print(f"\n✅ Successful: {successful}/{len(results)}")
print(f"❌ Failed: {failed}/{len(results)}")
print(f"📊 Total documents: {total_docs:,}")

if failed == 0:
    print("\n🎉 All collections loaded successfully!")
else:
    print(f"\n⚠️ {failed} collections failed to load")

## 4. Test Data Quality

In [ ]:
# Check data quality of first successful collection
for collection, result in results.items():
    if "OK" in result["status"]:
        print(f"\n🔍 Testing data quality for {collection}...")
        
        ds = load_dataset(
            "json",
            data_files=f"hf://datasets/{REPO_ID}/collections/{collection}.jsonl.gz",
            split="train"
        )
        
        # Check fields
        sample = ds[0]
        print(f"  Fields: {list(sample.keys())}")
        
        # Check for required fields
        required_fields = ["book_id", "title", "author", "content"]
        for field in required_fields:
            if field in sample:
                print(f"  ✅ {field}: {str(sample[field])[:80]}...")
            else:
                print(f"  ❌ Missing field: {field}")
        
        break

## 5. Ready for Embedding

If all tests pass, the datasets are ready for embedding. Next steps:

1. Run embedding script (see `notebooks/01_embed_all_collections.ipynb`)
2. Import embeddings to Qdrant
3. Test RAG retrieval